In [ ]:
import pandas as pd

db_uri = "sqlite:////Users/eidens/Projects/mecadoi-archives/batch/batch.sqlite3"
dry_run = True

In [ ]:
query = """
SELECT *
FROM deposition_attempt
WHERE deposition LIKE '%https://orcid.org/http://orcid.org%'
"""

df = pd.read_sql_query(query, db_uri)
df

In [ ]:
to_fix = df.iloc[9:]  # adjust as needed
to_fix

In [ ]:
from difflib import unified_diff

original_depositions = to_fix['deposition'].tolist()
corrected_depositions = [
    deposition.replace('https://orcid.org/http://orcid.org', 'https://orcid.org')
    for deposition in original_depositions
]
for i, (original, fixed) in enumerate(zip(original_depositions, corrected_depositions)):

    print(f'Deposition {i}:')

    dois_affected = [line for line in original.splitlines() if "<doi>" in line.lower()]
    for line in dois_affected:
        print(line)

    diff = unified_diff(original.splitlines(), fixed.splitlines())
    print('\n'.join(diff))

    print()

In [ ]:
from mecadoi.crossref.api import deposit

responses = []
for deposition in corrected_depositions:
    if dry_run:
        print(f"dry run - not depositing corrected deposition: {deposition[:60]}...")
        continue
    response = deposit(deposition, verbose=1)
    responses.append(response)
responses

In [ ]:
# write deposition attempt fixes back to the database?